# 半监督 GNN 空间转录组细胞类型注释

**流程**：scRNA（有标签）+ Xenium（无标签）联合训练，Domain Adaptation

**修复清单**：P0①②③ / P1④⑤⑥⑦ / P2⑧⑨⑩ / P3⑪⑫⑬

In [1]:
# ============================================================
# Cell 1: 环境配置
# ============================================================

import os, sys, warnings
import numpy as np
import pandas as pd
import torch
import rpy2.robjects as ro

warnings.filterwarnings("ignore")

# 路径管理
PATHS = {
    "cache": {
        "flex":   "./data/cache/flex_data_processed.rds",
        "xenium": "./data/cache/xenium_data_processed.rds",
    },
    "output": {
        "models":      "./results/models/",
        "predictions": "./results/predictions/",
        "plots":       "./plots/",
    },
}
for d in PATHS["output"].values():
    os.makedirs(d, exist_ok=True)

# ──── 全局超参（所有可调参数集中管理）────────────────────────────────
PARAMS = {
    # 图构建
    "knn_k":          30,
    "val_ratio":      0.2,

    # 模型架构
    "hidden_dim":     256,
    "proj_dim":       512,      # P3-⑭：投影层（3000→512→256），缓解跨度
    "dropout":        0.5,

    # 训练
    "n_epochs":       500,      # 半监督需要更多 epoch
    "lr":             1e-3,
    "weight_decay":   5e-4,
    "patience":       40,       # P2：patience 放宽

    # LR 调度器（P2-⑨）
    "lr_factor":      0.5,
    "lr_patience":    15,
    "min_lr":         1e-5,

    # 梯度裁剪（P2-⑩）
    "max_grad_norm":  1.0,

    # 热身阶段（仅 CE + MMD，不加伪标签）
    "warmup_epochs":  30,

    # Domain Adaptation 权重（P1-⑤）
    "lambda_mmd":     0.1,
    "lambda_ent":     0.01,
    "lambda_pl":      0.3,

    # 伪标签阈值（P3-⑫）
    "pl_threshold":   0.90,
    "pl_update_freq": 10,

    # 硬件
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed":   42,
}

from utils import set_seed
set_seed(PARAMS["seed"])

print(f"Device: {PARAMS['device']}")
print(f"PyTorch: {torch.__version__}")
print("Params configured ✓")


Device: cuda
PyTorch: 2.3.1
Params configured ✓


## Step 1: 数据加载（R → Python）

从 `labelTransfer.ipynb` 生成的缓存中加载数据，取 Xenium panel 共同基因。

In [2]:
%load_ext rpy2.ipython

In [3]:
%%R -o flex_expr -o xenium_expr -o flex_labels -o cell_types -o xenium_coords
# ============================================================
# Cell 2: 数据加载（%%R 魔法命令）
# ============================================================
library(Seurat)

cat("Loading cached data...\n")
flex_data   <- readRDS("./data/cache/flex_data_processed.rds")
xenium_data <- readRDS("./data/cache/xenium_data_processed.rds")

# ── 特征对齐：只使用 Xenium panel 基因（P0 核心要求）──────────────
# 错误做法：intersect(rownames(flex_data), rownames(xenium_data)) 后用 scRNA 全基因
# 正确做法：以 Xenium panel 为基准，scRNA 也只看这些基因
xenium_panel <- rownames(xenium_data)
common_genes <- intersect(rownames(flex_data), xenium_panel)
cat("Xenium panel:", length(xenium_panel), "| Common genes:", length(common_genes), "\n")

# ── 提取原始 counts（log_normalize 在 Python 做，保持一致性）─────
flex_counts   <- as.matrix(GetAssayData(flex_data,   layer="counts")[common_genes, ])
xenium_counts <- as.matrix(GetAssayData(xenium_data, layer="counts")[common_genes, ])

# 转置为 (n_cells, n_genes)
flex_expr   <- t(flex_counts)
xenium_expr <- t(xenium_counts)

# 细胞类型标签
flex_labels_raw <- flex_data$cell_type
unique_labels   <- unique(flex_labels_raw)
label_map       <- setNames(seq_along(unique_labels) - 1L, unique_labels)
flex_labels     <- as.integer(label_map[flex_labels_raw])
cell_types      <- unique_labels

# Xenium 空间坐标（μm）
xenium_coords <- as.matrix(data.frame(
    x = xenium_data@images[[1]]@boundaries$centroids@coords[, 1],
    y = xenium_data@images[[1]]@boundaries$centroids@coords[, 2]
))

cat("scRNA cells:", nrow(flex_expr), "| Xenium cells:", nrow(xenium_expr), "\n")
cat("Cell types:", length(cell_types), "\n")
cat(cell_types, sep="  ")


R[write to console]: Loading required package: SeuratObject

R[write to console]: Loading required package: sp

R[write to console]: 
Attaching package: ‘SeuratObject’


R[write to console]: The following objects are masked from ‘package:base’:

    intersect, t





    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    Loading cached data...
Xenium panel: 5101 | Common genes: 4912 


R[write to console]: Loading required package: BPCells



This message is displayed once every 8 hours.
scRNA cells: 17050 | Xenium cells: 406611 
Cell types: 16 
Tumor Associated Fibroblasts  Endothelial Cells  Stromal Associated Fibroblasts  T & NK Cells  Malignant Cells Lining Cyst  Proliferative Tumor Cells  Tumor Cells  Pericytes  Granulosa Cells  Macrophages  Smooth Muscle Cells  VEGFA+ Tumor Cells  MT-High, Jun+/Fos+ Tumor Cells  Fallopian Tube Epithelium  Inflammatory Tumor Cells  Ciliated Epithelial Cells

## Step 2: 数据预处理

修复 P0-① 和 P1-⑥：统一归一化 + 生物学标准化

In [4]:
# ============================================================
# Cell 3: 预处理 + 联合图构建（支持磁盘缓存）
# ============================================================
import os, pickle
import numpy as np
import torch
from utils import log_normalize, unified_normalize, build_combined_dataset

# 缓存路径（PARAMS 里的 knn_k 变了就自动重建）
GRAPH_CACHE_DIR  = "./data/cache/graph/"
os.makedirs(GRAPH_CACHE_DIR, exist_ok=True)

CACHE_FILE = os.path.join(
    GRAPH_CACHE_DIR,
    f"graph_k{PARAMS['knn_k']}_val{PARAMS['val_ratio']}.pt"
)
SCALER_FILE = os.path.join(GRAPH_CACHE_DIR, "fitted_scaler.pkl")
LOG_FILE    = os.path.join(GRAPH_CACHE_DIR, "log_normalized.npz")

def _build_and_save():
    """从原始数据构建图并保存到磁盘。"""
    print("  [Cache MISS] Building graph from scratch...")

    # ── P1-⑥ 生物学标准化 ─────────────────────────────────────────
    print("  Step 1: log1p normalization...")
    flex_log_   = log_normalize(flex_expr)
    xenium_log_ = log_normalize(xenium_expr)

    # ── P0-① 统一归一化（只在 scRNA 上 fit）──────────────────────
    print("  Step 2: unified StandardScaler (fit on scRNA only)...")
    flex_norm_, xenium_norm_, scaler_ = unified_normalize(flex_log_, xenium_log_)

    # ── P1-⑦⑧ 构建联合图 ─────────────────────────────────────────
    print("  Step 3: building mutual kNN graph...")
    data_, cw_, split_ = build_combined_dataset(
        flex_norm_, xenium_norm_, flex_labels,
        k=PARAMS["knn_k"],
        val_ratio=PARAMS["val_ratio"],
    )

    # ── 保存 ──────────────────────────────────────────────────────
    torch.save({
        "data":         data_,
        "class_weights": cw_,
        "split_info":   split_,
    }, CACHE_FILE)

    with open(SCALER_FILE, "wb") as f:
        pickle.dump(scaler_, f)

    np.savez_compressed(
        LOG_FILE,
        flex_log=flex_log_,
        xenium_log=xenium_log_,
    )

    print(f"  Saved graph cache → {CACHE_FILE}")
    return data_, cw_, split_, scaler_, flex_log_, xenium_log_


def _load_cache():
    """从磁盘加载已缓存的图和预处理结果。"""
    print("  [Cache HIT] Loading graph from disk...")
    ckpt = torch.load(CACHE_FILE, weights_only=False)
    data_    = ckpt["data"]
    cw_      = ckpt["class_weights"]
    split_   = ckpt["split_info"]

    with open(SCALER_FILE, "rb") as f:
        scaler_ = pickle.load(f)

    arrays     = np.load(LOG_FILE)
    flex_log_  = arrays["flex_log"]
    xenium_log_= arrays["xenium_log"]

    print(f"  Loaded: {data_.x.shape[0]:,} nodes | "
          f"{data_.edge_index.shape[1]:,} edges")
    return data_, cw_, split_, scaler_, flex_log_, xenium_log_


# ── 判断是否命中缓存（三个文件都存在才算命中）────────────────────────
CACHE_EXISTS = all(os.path.exists(p)
                   for p in [CACHE_FILE, SCALER_FILE, LOG_FILE])

# 强制重建开关：把下面改成 True 可跳过缓存强制重建
FORCE_REBUILD = False

if CACHE_EXISTS and not FORCE_REBUILD:
    data, class_weights, split_info, fitted_scaler, flex_log, xenium_log = _load_cache()
else:
    data, class_weights, split_info, fitted_scaler, flex_log, xenium_log = _build_and_save()

print(f"\nClass weights : {class_weights.numpy().round(3)}")
print(f"Train / Val   : {data.train_mask.sum()} / {data.val_mask.sum()}")
print(f"Feature dim   : {data.x.shape[1]}")
print("Preprocessing complete ✓")


  [Cache MISS] Building graph from scratch...
  Step 1: log1p normalization...
  Step 2: unified StandardScaler (fit on scRNA only)...
  Step 3: building mutual kNN graph...
  Building mutual kNN graph on 423,661 cells (k=30)…
  Edges: 1,542,051 | Train: 13,640 | Val: 3,410 | Xenium: 406,611
  Saved graph cache → ./data/cache/graph/graph_k30_val0.2.pt

Class weights : [ 0.473  0.788  0.775  1.312 10.874  0.546  0.473  1.21   4.163  0.458
  1.111  0.804  1.758  3.576  7.894  6.305]
Train / Val   : 13640 / 3410
Feature dim   : 4912
Preprocessing complete ✓


## Step 3: 训练 GNN 模型

每个模型独立运行，失败不影响其他模型。
`gnn_results` 字典会在各 cell 运行后逐步填充。

## Step 3a: 训练 GCN

In [5]:
# ============================================================
# Cell 4a: 训练 GCN
# ============================================================
import torch
from models import GCN, run_experiment

if "gnn_results" not in dir():
    gnn_results = {}

result_gcn = run_experiment(
    GCN, "GCN",
    data, list(cell_types), PARAMS, class_weights,
)
gnn_results["GCN"] = result_gcn

torch.save(
    result_gcn["model"].state_dict(),
    f"{PATHS['output']['models']}/GCN_best.pt"
)
print("\n✅ GCN training complete.")
print(f"   Best val F1-macro: {result_gcn['best_val_f1']:.4f}")



 GCN
  Ep   1 | CE=3.078 MMD=0.0195 Ent=2.2548 PL=0.0000 | Val Acc=0.268 F1=0.190 lr=1.00e-03
  Ep   2 | CE=2.398 MMD=0.0151 Ent=2.2687 PL=0.0000 | Val Acc=0.404 F1=0.336 lr=1.00e-03
  Ep   3 | CE=1.954 MMD=0.0082 Ent=2.2509 PL=0.0000 | Val Acc=0.503 F1=0.424 lr=1.00e-03
  Ep   4 | CE=1.664 MMD=0.0065 Ent=2.2098 PL=0.0000 | Val Acc=0.605 F1=0.540 lr=1.00e-03
  Ep   5 | CE=1.322 MMD=0.0074 Ent=2.1577 PL=0.0000 | Val Acc=0.666 F1=0.612 lr=1.00e-03
  Ep  20 | CE=0.176 MMD=0.0469 Ent=1.5141 PL=0.0000 | Val Acc=0.848 F1=0.797 lr=1.00e-03
  [Epoch  31] Pseudo-labels updated: 80,910 cells (19.9%)
  Ep  40 | CE=0.080 MMD=0.0452 Ent=1.3077 PL=0.0590 | Val Acc=0.840 F1=0.792 lr=5.00e-04
  [Epoch  41] Pseudo-labels updated: 108,951 cells (26.8%)
  [Epoch  51] Pseudo-labels updated: 130,709 cells (32.1%)
  Early stopping at epoch 59 (best F1=0.8009)
  Best val F1-macro: 0.8009

✅ GCN training complete.
   Best val F1-macro: 0.8009


## Step 3b: 训练 GraphSAGE

In [6]:
# ============================================================
# Cell 4b: 训练 GraphSAGE
# ============================================================
import torch
from models import GraphSAGE, run_experiment

if "gnn_results" not in dir():
    gnn_results = {}

result_sage = run_experiment(
    GraphSAGE, "GraphSAGE",
    data, list(cell_types), PARAMS, class_weights,
)
gnn_results["GraphSAGE"] = result_sage

torch.save(
    result_sage["model"].state_dict(),
    f"{PATHS['output']['models']}/GraphSAGE_best.pt"
)
print("\n✅ GraphSAGE training complete.")
print(f"   Best val F1-macro: {result_sage['best_val_f1']:.4f}")



 GraphSAGE
  Ep   1 | CE=2.825 MMD=0.0147 Ent=2.5990 PL=0.0000 | Val Acc=0.330 F1=0.218 lr=1.00e-03
  Ep   2 | CE=2.429 MMD=0.0068 Ent=2.5967 PL=0.0000 | Val Acc=0.414 F1=0.322 lr=1.00e-03
  Ep   3 | CE=2.106 MMD=0.0047 Ent=2.5701 PL=0.0000 | Val Acc=0.470 F1=0.387 lr=1.00e-03
  Ep   4 | CE=1.910 MMD=0.0100 Ent=2.5380 PL=0.0000 | Val Acc=0.617 F1=0.538 lr=1.00e-03
  Ep   5 | CE=1.671 MMD=0.0163 Ent=2.5158 PL=0.0000 | Val Acc=0.668 F1=0.589 lr=1.00e-03
  Ep  20 | CE=0.135 MMD=0.1018 Ent=2.2596 PL=0.0000 | Val Acc=0.883 F1=0.857 lr=1.00e-03
  [Epoch  31] Pseudo-labels updated: 11,670 cells (2.9%)
  Ep  40 | CE=0.012 MMD=0.1004 Ent=1.9465 PL=0.0159 | Val Acc=0.880 F1=0.849 lr=5.00e-04
  [Epoch  41] Pseudo-labels updated: 60,587 cells (14.9%)
  [Epoch  51] Pseudo-labels updated: 91,592 cells (22.5%)
  Ep  60 | CE=0.009 MMD=0.0935 Ent=1.6700 PL=0.0569 | Val Acc=0.882 F1=0.852 lr=2.50e-04
  [Epoch  61] Pseudo-labels updated: 108,548 cells (26.7%)
  Early stopping at epoch 65 (best F1=0.8634

## Step 3c: 训练 GAT

In [ ]:
# Cell 4c: 训练 GAT（AMP 显存优化版）
import torch
from models_amp import run_gat_amp

# 在 PARAMS 里加入这三个开关
PARAMS["use_amp"] = True  # 混合精度，显存减少约 45%
PARAMS["gat_heads"] = 4  # 如果还 OOM 就改成 2
PARAMS["use_ckpt"] = False  # 最后手段：梯度检查点（再减 30%，但慢 20%）

if "gnn_results" not in dir():
    gnn_results = {}

result_gat = run_gat_amp(data, list(cell_types), PARAMS, class_weights)
gnn_results["GAT"] = result_gat

torch.save(result_gat["model"].state_dict(), f"{PATHS['output']['models']}/GAT_best.pt")
print(f"\n✅ GAT 训练完成  Best val F1: {result_gat['best_val_f1']:.4f}")


 GAT


OutOfMemoryError: CUDA out of memory. Tried to allocate 7.50 GiB. GPU 

## Step 3d: GNN 训练汇总

In [ ]:
# ============================================================
# Cell 4d: GNN 训练汇总
# ============================================================
print("=" * 50)
print("  GNN Training Summary")
print("=" * 50)
for name, res in gnn_results.items():
    best_ep = max(range(len(res["history"])),
                  key=lambda i: res["history"][i]["val_f1_macro"])
    print(f"  {name:<12} val F1-macro = {res['best_val_f1']:.4f} "
          f"(best epoch: {res['history'][best_ep]['epoch']})")
print("=" * 50)
print(f"\nModels trained: {list(gnn_results.keys())}")
print("\n⚠️  如果某个模型尚未运行，请先运行对应的 Cell 4a/4b/4c")


## Step 4: TopACT Baseline（SVM + 空间平滑）

In [ ]:
# ============================================================
# Cell 5: TopACT
# ============================================================

from topact import TopACT

topact = TopACT(
    C=10.0,
    gamma="scale",
    kernel="rbf",
    spatial_weight=0.3,
    n_neighbors=10,
)

# 使用与 GNN 相同的 scaler（特征对齐）
print("Fitting TopACT SVM on scRNA data...")
topact.fit(flex_log, flex_labels, fitted_scaler=fitted_scaler)

# 推断 Xenium
print("Predicting Xenium cell types...")
topact_pred_idx, topact_proba = topact.predict(
    xenium_log,
    spatial_coords=xenium_coords,
    return_proba=True,
)
topact_confidence = topact_proba.max(axis=1)

topact_results = {
    "pred_indices": topact_pred_idx,
    "labels":       [cell_types[i] for i in topact_pred_idx],
    "probs":        topact_proba,
    "confidence":   topact_confidence,
}

print(f"TopACT prediction complete. "
      f"Median confidence: {topact_confidence.median():.3f}")


## Step 5: 评估（scRNA 验证集 — 唯一合法的 ground truth）

> ⚠️ **P0-④**：Seurat 标签只用于「事后对比分析」，不参与模型选择/调参。

In [ ]:
# ============================================================
# Cell 6: 评估（scRNA 验证集）
# ============================================================

import torch
import torch.nn.functional as F
from sklearn.metrics import classification_report
from eval import compute_metrics

val_idx    = split_info["val_idx"]
val_true   = flex_labels[val_idx]

# ── 获取每个方法在验证集上的预测 ────────────────────────────────────
val_preds = {}

for name, res in gnn_results.items():
    model  = res["model"].to("cpu")
    data_c = data.to("cpu")
    model.eval()
    with torch.no_grad():
        log_probs = model(data_c)
    val_pred_idx = log_probs[data_c.val_mask].argmax(dim=1).numpy()
    val_preds[name] = val_pred_idx

# TopACT 验证集预测
topact_val_pred, _ = topact.predict(flex_log[val_idx], return_proba=True)
val_preds["TopACT"] = topact_val_pred

# ── 打印指标 ─────────────────────────────────────────────────────────
print("=" * 65)
print("  Method Comparison on scRNA Val Set (Real Ground Truth)")
print("=" * 65)
print(f"{'Method':<12} {'Acc':>7} {'F1-Mac':>8} {'F1-Wei':>8} {'Kappa':>8}")
print("-" * 65)

for method, y_pred in val_preds.items():
    m = compute_metrics(val_true, y_pred, n_classes=len(cell_types))
    print(f"{method:<12} {m['accuracy']:>7.4f} {m['f1_macro']:>8.4f} "
          f"{m['f1_weighted']:>8.4f} {m['kappa']:>8.4f}")
    if method == list(val_preds.keys())[-1]:
        print("=" * 65)

# ── 最佳模型详细报告 ─────────────────────────────────────────────────
best_method = max(val_preds,
                  key=lambda m: compute_metrics(val_true, val_preds[m])["f1_macro"]
                  if m in gnn_results else 0)
print(f"\nBest GNN: {best_method}")
print(classification_report(
    val_true, val_preds[best_method],
    target_names=list(cell_types), zero_division=0
))


## Step 6: Moran's I 空间自相关评估

In [ ]:
# ============================================================
# Cell 7: Moran's I（空间自相关）
# ============================================================

from topact import TopACT

morans_results = {}

all_xenium_preds = {name: res["pred_indices"] for name, res in gnn_results.items()}
all_xenium_preds["TopACT"] = topact_results["pred_indices"]

for method, pred_idx in all_xenium_preds.items():
    print(f"Computing Moran's I for {method}...", end=" ")
    global_mi = TopACT.morans_i(pred_idx, xenium_coords, n_neighbors=15)
    per_class  = TopACT.per_class_morans_i(
        pred_idx, xenium_coords, list(cell_types), n_neighbors=15
    )
    morans_results[method] = per_class
    print(f"Global I = {global_mi['I']:.4f}  (p={global_mi['p_value']:.4f})")

print("\nMoran's I computation complete ✓")


## Step 7: 生成所有论文图表（Fig 1–10）

In [ ]:
# ============================================================
# Cell 8: 生成全部论文图表
# ============================================================

from eval import generate_all_thesis_figures

# 为 Fig 2 (UMAP) 准备 scRNA 隐层嵌入
import torch

scrna_embeddings_per_model = {}
for name, res in gnn_results.items():
    model  = res["model"].to("cpu")
    data_c = data.to("cpu")
    model.eval()
    with torch.no_grad():
        h, _ = model.encode(data_c)
    # 前 n_flex 行是 scRNA，后面是 Xenium
    n_flex = data_c.n_flex
    scrna_h  = h[:n_flex].numpy()
    xenium_h = h[n_flex:].numpy()
    # 把两部分合并放入 embeddings（eval.py 会根据 n_flex 切分）
    gnn_results[name]["embeddings"] = np.vstack([scrna_h, xenium_h])

generate_all_thesis_figures(
    gnn_results        = gnn_results,
    topact_results     = topact_results,
    scrna_expr_norm    = flex_norm,
    scrna_labels       = flex_labels,
    xenium_coords      = xenium_coords,
    cell_types         = list(cell_types),
    val_labels         = val_true,
    val_pred_per_method= val_preds,
    morans_results     = morans_results,
    output_dir         = PATHS["output"]["plots"],
)


## Step 8: 保存最终预测结果

In [ ]:
# ============================================================
# Cell 9: 保存结果
# ============================================================

import pandas as pd, json

# ── 构建预测 DataFrame ──────────────────────────────────────────────
pred_df = pd.DataFrame(index=range(data.n_xenium))

for name, res in gnn_results.items():
    pred_df[f"{name}_label"]      = res["predictions"]
    pred_df[f"{name}_confidence"] = res["confidence"]
    proba_cols = pd.DataFrame(
        res["probabilities"],
        columns=[f"{name}_prob_{ct}" for ct in cell_types]
    )
    pred_df = pd.concat([pred_df, proba_cols], axis=1)

pred_df["TopACT_label"]      = topact_results["labels"]
pred_df["TopACT_confidence"] = topact_results["confidence"]

if xenium_coords is not None:
    pred_df["x_coord"] = xenium_coords[:, 0]
    pred_df["y_coord"] = xenium_coords[:, 1]

out_csv = f"{PATHS['output']['predictions']}/all_predictions.csv"
pred_df.to_csv(out_csv, index=False)
print(f"Predictions saved → {out_csv}")

# ── 实验摘要 ─────────────────────────────────────────────────────────
summary = {
    "cell_types":   list(cell_types),
    "n_scrna":      int(data.n_flex),
    "n_xenium":     int(data.n_xenium),
    "n_genes":      int(data.x.shape[1]),
    "params":       {k: str(v) for k, v in PARAMS.items()},
    "val_metrics":  {},
}

from eval import compute_metrics
for method, y_pred in val_preds.items():
    m = compute_metrics(val_true, y_pred, n_classes=len(cell_types))
    summary["val_metrics"][method] = {
        "accuracy":    round(float(m["accuracy"]),    4),
        "f1_macro":    round(float(m["f1_macro"]),    4),
        "f1_weighted": round(float(m["f1_weighted"]), 4),
        "kappa":       round(float(m["kappa"]),       4),
    }

summary_path = f"{PATHS['output']['predictions']}/experiment_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"Summary saved    → {summary_path}")

print("\n" + "="*60)
print("  实验完成！")
print("="*60)
print(f"  图表目录 : {PATHS['output']['plots']}")
print(f"  预测文件 : {out_csv}")
print(f"  摘要文件 : {summary_path}")


## (可选) 事后分析：与 Seurat 标签对比

> ⚠️ **仅用于定性对比，不参与调参或模型选择（P0-④）。**

In [ ]:
# ============================================================
# Cell 10: 事后 Seurat 对比（可选，不影响主实验）
# ============================================================

try:
    import rpy2.robjects as ro
    from rpy2.robjects import pandas2ri
    pandas2ri.activate()

    ro.r("""
        obj  <- readRDS('./results/seurat/xenium_annotated_final.rds')
        seurat_labels <- as.character(obj$predicted.id)
    """)
    seurat_labels_raw = list(ro.r("seurat_labels"))

    # 将字符串标签映射为整数
    ct_list  = list(cell_types)
    ct2idx   = {ct: i for i, ct in enumerate(ct_list)}
    seurat_idx = np.array([ct2idx.get(l, -1) for l in seurat_labels_raw])
    valid      = seurat_idx >= 0
    print(f"Seurat labels loaded: {valid.sum()} / {len(seurat_idx)} valid")

    print("\n─── Agreement with Seurat (qualitative only) ───")
    for method, pred_idx in {
        **{n: r["pred_indices"] for n, r in gnn_results.items()},
        "TopACT": topact_results["pred_indices"],
    }.items():
        agree = (pred_idx[valid] == seurat_idx[valid]).mean()
        print(f"  {method:<12}: agreement = {agree:.4f}")

    print("\n⚠️  上述数字仅供参考，不代表真实准确率（Seurat 本身也有噪声）。")

except Exception as e:
    print(f"Seurat comparison skipped ({e})")
